In [1]:
import numpy as np
import math
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
from scipy.stats import norm

# =========================================================
# 1. Data: 6D inputs and 1D outputs (Function 7)
# =========================================================

X_raw = np.array([
    [0.27262382, 0.32449536, 0.89710881, 0.83295115, 0.15406269, 0.79586362],
    [0.54300258, 0.9246939 , 0.34156746, 0.64648585, 0.71844033, 0.34313266],
    [0.09083225, 0.66152938, 0.06593091, 0.25857701, 0.96345285, 0.6402654 ],
    [0.11886697, 0.61505494, 0.90581639, 0.8553003 , 0.41363143, 0.58523563],
    [0.63021764, 0.8380969 , 0.68001305, 0.73189509, 0.52673671, 0.34842921],
    [0.76491917, 0.25588292, 0.60908422, 0.21807904, 0.32294277, 0.09579366],
    [0.05789554, 0.49167222, 0.24742222, 0.21811844, 0.42042833, 0.73096984],
    [0.19525188, 0.07922665, 0.55458046, 0.17056682, 0.01494418, 0.10703171],
    [0.64230298, 0.83687455, 0.02179269, 0.10148801, 0.68307083, 0.6924164 ],
    [0.78994255, 0.19554501, 0.57562333, 0.07365919, 0.25904917, 0.05109986],
    [0.52849733, 0.45742436, 0.36009569, 0.36204551, 0.81689098, 0.63747637],
    [0.72261522, 0.01181284, 0.06364591, 0.16517311, 0.07924415, 0.35995166],
    [0.07566492, 0.33450212, 0.13273274, 0.60831236, 0.91838592, 0.82233079],
    [0.94245084, 0.37743962, 0.48612233, 0.22879108, 0.08263175, 0.71195755],
    [0.14864702, 0.03394336, 0.72880565, 0.31606646, 0.02176938, 0.51691776],
    [0.81711239, 0.54816823, 0.10334758, 0.12436955, 0.72823482, 0.44967361],
    [0.41762629, 0.06409998, 0.24566877, 0.5590408 , 0.19153138, 0.25464092],
    [0.72628566, 0.46489581, 0.92457051, 0.8072454 , 0.6354384 , 0.14341787],
    [0.31981043, 0.52009759, 0.29067775, 0.87670668, 0.49503469, 0.6190825 ],
    [0.87987128, 0.39796199, 0.00363456, 0.95699064, 0.26451373, 0.11486924],
    [0.54124078, 0.63140314, 0.03190205, 0.44998156, 0.79865282, 0.63370429],
    [0.22634792, 0.11502581, 0.82474966, 0.94538372, 0.90531153, 0.95101392],
    [0.68685257, 0.04101721, 0.00757301, 0.285009  , 0.69156848, 0.6555429 ],
    [0.17597754, 0.6244165 , 0.29554198, 0.46955276, 0.09776977, 0.72814108],
    [0.88164674, 0.20445019, 0.41447436, 0.42038468, 0.26491501, 0.73066019],
    [0.06661051, 0.52804507, 0.8160952 , 0.96101714, 0.08650933, 0.77778822],
    [0.93246638, 0.48881189, 0.25860774, 0.95624344, 0.19042781, 0.51985176],
    [0.84686697, 0.14242917, 0.06066859, 0.75629213, 0.5523983 , 0.08130609],
    [0.80628208, 0.32412237, 0.72607601, 0.14871213, 0.7193764 , 0.36288398],
    [0.47682313, 0.34094195, 0.01433523, 0.88013956, 0.9986547 , 0.07966402],
    [1.04245   , 1.024693  , 1.02457   , 1.061017  , 1.098654  , 1.051013  ],
    [0.019976  , 0.432955  , 0.301662  , 0.169496  , 0.348651  , 0.743371  ],
    [0.611853  , 0.139495  , 0.292145  , 0.366362  , 0.45607   , 0.785175  ],
    [0.015006  , 0.390905  , 0.178469  , 0.119929  , 0.088415  , 0.904408  ]
])

y_raw = np.array([
    6.04432696e-01, 5.62753067e-01, 7.50323668e-03, 6.14243025e-02,
    2.73046801e-01, 8.37465723e-02, 1.36496830e+00, 9.26449549e-02,
    1.78695987e-02, 3.35649360e-02, 7.35163042e-02, 2.06309698e-01,
    8.82563400e-03, 2.68400317e-01, 6.11525528e-01, 1.47981826e-02,
    2.74892508e-01, 6.67632469e-02, 4.21183545e-02, 2.70146502e-03,
    1.82090730e-02, 7.01602756e-03, 1.00506611e-01, 4.75395516e-01,
    6.75141631e-01, 5.16457219e-01, 3.77747962e-03, 3.13433331e-03,
    2.13425228e-02, 9.54111589e-02, 4.63685805e-06, 1.68082842e+00,
    1.11705767e+00, 4.40998916e-01
])

# =========================================================
# 2. Configuration
# =========================================================

RANDOM_STATE = 123
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

INPUT_DIM = X_raw.shape[1]
BOUNDS_LOWER = np.zeros(INPUT_DIM)
BOUNDS_UPPER = np.ones(INPUT_DIM)

N_CANDIDATES = 20000      # random candidate points for EI search
XI = 0.01                 # exploration parameter for EI
N_ENSEMBLE = 25           # number of neural nets in the ensemble
N_EPOCHS = 1500           # training epochs per model
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
HIDDEN_SIZES = (128, 64)
DEVICE = torch.device("cpu")

# =========================================================
# 3. PyTorch MLP definition
# =========================================================

class MLPRegressorTorch(nn.Module):
    def __init__(self, input_dim, hidden_sizes):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for h in hidden_sizes:
            layers.append(nn.Linear(prev_dim, h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(p=0.1))  # small dropout for stability
            prev_dim = h
        layers.append(nn.Linear(prev_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

# =========================================================
# 4. Acquisition functions (Gaussian PI / EI)
# =========================================================

def gaussian_pi(mu, sigma, f_best, xi=0.0):
    sigma = np.maximum(sigma, 1e-9)
    z = (mu - f_best - xi) / sigma
    return norm.cdf(z)

def gaussian_ei(mu, sigma, f_best, xi=0.0):
    sigma = np.maximum(sigma, 1e-9)
    z = (mu - f_best - xi) / sigma
    ei = (mu - f_best - xi) * norm.cdf(z) + sigma * norm.pdf(z)
    ei[sigma < 1e-9] = 0.0
    return ei

# =========================================================
# 5. Train an ensemble of PyTorch MLPs (bagging-style)
# =========================================================

def train_single_model(X_scaled, y_scaled, random_state):
    """Train one MLP on a bootstrap sample of the data."""
    torch.manual_seed(random_state)
    n_samples, n_features = X_scaled.shape

    # Bootstrap sample indices
    rng = np.random.RandomState(random_state)
    indices = rng.randint(0, n_samples, size=n_samples)
    X_boot = X_scaled[indices]
    y_boot = y_scaled[indices]

    # Convert to tensors
    X_tensor = torch.tensor(X_boot, dtype=torch.float32, device=DEVICE)
    y_tensor = torch.tensor(y_boot.reshape(-1, 1), dtype=torch.float32, device=DEVICE)

    model = MLPRegressorTorch(input_dim=n_features, hidden_sizes=HIDDEN_SIZES).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    criterion = nn.MSELoss()

    model.train()
    for epoch in range(N_EPOCHS):
        optimizer.zero_grad()
        preds = model(X_tensor)
        loss = criterion(preds, y_tensor)
        loss.backward()
        optimizer.step()

    return model

def build_ensemble_torch(X_scaled, y_scaled):
    ensemble = []
    for i in range(N_ENSEMBLE):
        seed = RANDOM_STATE + i
        model = train_single_model(X_scaled, y_scaled, random_state=seed)
        ensemble.append(model)
    return ensemble

# =========================================================
# 6. Propose next query point using EI with PyTorch ensemble
# =========================================================

def propose_next_point(ensemble, x_scaler, y_scaler,
                       X_raw, y_raw,
                       n_candidates=N_CANDIDATES,
                       xi=XI,
                       random_state=RANDOM_STATE):
    rng = np.random.RandomState(random_state)

    # Current best from observed data (true outputs)
    f_best = np.max(y_raw)
    idx_best = np.argmax(y_raw)
    x_best = X_raw[idx_best]

    # Sample candidate points uniformly within [0,1]^d
    X_cand = rng.uniform(low=BOUNDS_LOWER, high=BOUNDS_UPPER,
                         size=(n_candidates, INPUT_DIM))

    # Scale candidates
    X_cand_scaled = x_scaler.transform(X_cand)
    X_cand_tensor = torch.tensor(X_cand_scaled, dtype=torch.float32, device=DEVICE)

    # Collect ensemble predictions (on scaled y)
    preds_scaled_list = []
    for model in ensemble:
        model.eval()
        with torch.no_grad():
            y_pred = model(X_cand_tensor).cpu().numpy().ravel()
        preds_scaled_list.append(y_pred)

    preds_scaled = np.stack(preds_scaled_list, axis=0)  # (n_ensemble, n_candidates)
    preds_scaled = preds_scaled.T                        # (n_candidates, n_ensemble)

    # Inverse-transform to original y scale
    preds_unscaled = y_scaler.inverse_transform(preds_scaled)  # (n_candidates, n_ensemble)
    mu = preds_unscaled.mean(axis=1)
    sigma = preds_unscaled.std(axis=1, ddof=1)

    # Compute acquisition
    pi = gaussian_pi(mu, sigma, f_best, xi=0.0)
    ei = gaussian_ei(mu, sigma, f_best, xi=xi)

    # Select best candidate
    best_idx = np.argmax(ei)
    x_next = X_cand[best_idx]
    mu_next = mu[best_idx]
    sigma_next = sigma[best_idx]
    pi_next = pi[best_idx]
    ei_next = ei[best_idx]

    info = {
        "x_best_observed": x_best,
        "f_best_observed": f_best,
        "x_next": x_next,
        "mu_next": mu_next,
        "sigma_next": sigma_next,
        "pi_next": pi_next,
        "ei_next": ei_next
    }
    return info

# =========================================================
# 7. Human-readable explanation
# =========================================================

def explain_recommendation(info):
    x_best = info["x_best_observed"]
    f_best = info["f_best_observed"]
    x_next = info["x_next"]
    mu_next = info["mu_next"]
    sigma_next = info["sigma_next"]
    pi_next = info["pi_next"]
    ei_next = info["ei_next"]

    lines = []
    lines.append("=== Agent-style Reasoning for Next Query ===")
    lines.append(f"1. The current best observed output (local maxima so far) is f_best = {f_best:.6f}.")
    lines.append("   This comes from input:")
    lines.append("   x_best = " + np.array2string(x_best, precision=6))
    lines.append("")
    lines.append("2. I trained an ensemble of PyTorch neural networks (MLPs) as a deep surrogate model.")
    lines.append("   Each model is trained on a bootstrap resample of the data, which creates diversity")
    lines.append("   and allows the ensemble to approximate predictive uncertainty.")
    lines.append("")
    lines.append("3. I sampled many candidate points uniformly within the bounds [0, 1]^6 and")
    lines.append("   evaluated their Expected Improvement (EI) and Probability of Improvement (PI)")
    lines.append("   under a Gaussian approximation to the ensemble predictions.")
    lines.append("")
    lines.append("4. The recommended next query point is:")
    lines.append("   x_next = " + np.array2string(x_next, precision=6))
    lines.append("")
    lines.append("   At this point, the surrogate predicts:")
    lines.append(f"   - Predicted mean output (mu_next): {mu_next:.6f}")
    lines.append(f"   - Predictive standard deviation (sigma_next): {sigma_next:.6f}")
    lines.append(f"   - Probability of beating current best (PI): {pi_next:.4f}")
    lines.append(f"   - Expected Improvement (EI): {ei_next:.6f}")
    lines.append("")
    if mu_next > f_best:
        lines.append("5. The predicted mean at x_next is higher than the current best,")
        lines.append("   which suggests this point could yield a better local maxima if evaluated.")
    else:
        lines.append("5. The predicted mean at x_next is slightly below the current best,")
        lines.append("   but the uncertainty is large enough that the Expected Improvement is still high.")
    lines.append("   The non-zero EI and PI reflect a trade-off between exploiting high predicted values")
    lines.append("   and exploring uncertain regions where a significantly better value might be found.")
    lines.append("")
    lines.append("6. Therefore, x_next is recommended as the next query point most likely to produce")
    lines.append("   an output higher than the current observed local maxima, given the PyTorch")
    lines.append("   neural-network ensemble surrogate and the observed data.")
    return "\n".join(lines)

# =========================================================
# 8. Main script
# =========================================================

def main():
    print("=== Data summary ===")
    print(f"Number of samples: {X_raw.shape[0]}")
    print(f"Input dimension:   {X_raw.shape[1]}")
    print()

    # Check input bounds
    x_min = X_raw.min(axis=0)
    x_max = X_raw.max(axis=0)
    print("Min of existing inputs (per dimension):", x_min)
    print("Max of existing inputs (per dimension):", x_max)
    print("Note: some inputs slightly exceed [0, 1];")
    print("      the search for the NEXT query will be restricted to [0, 1]^6.")
    print()

    # Current best observation
    idx_best = np.argmax(y_raw)
    x_best = X_raw[idx_best]
    f_best = y_raw[idx_best]
    print("=== Current best (from observed data) ===")
    print("x_best =", x_best)
    print("f(x_best) =", f_best)
    print()

    # Scale inputs and outputs
    x_scaler = StandardScaler()
    y_scaler = StandardScaler()

    X_scaled = x_scaler.fit_transform(X_raw)
    y_scaled = y_scaler.fit_transform(y_raw.reshape(-1, 1)).ravel()

    # Train PyTorch ensemble surrogate
    print("=== Training PyTorch ensemble surrogate ===")
    ensemble = build_ensemble_torch(X_scaled, y_scaled)
    print("Training complete.\n")

    # Propose next query point
    info = propose_next_point(
        ensemble, x_scaler, y_scaler,
        X_raw, y_raw,
        n_candidates=N_CANDIDATES,
        xi=XI,
        random_state=RANDOM_STATE
    )

    print("=== Proposed NEXT query point (within [0, 1]^6) ===")
    print("x_next =", info["x_next"])
    print(f"Predicted mean at x_next: {info['mu_next']:.6f}")
    print(f"Predictive std at x_next: {info['sigma_next']:.6f}")
    print(f"Probability of Improvement (PI): {info['pi_next']:.4f}")
    print(f"Expected Improvement (EI): {info['ei_next']:.6f}")
    print()

    reasoning_text = explain_recommendation(info)
    print(reasoning_text)

if __name__ == "__main__":
    main()


=== Data summary ===
Number of samples: 34
Input dimension:   6

Min of existing inputs (per dimension): [0.015006   0.01181284 0.00363456 0.07365919 0.01494418 0.05109986]
Max of existing inputs (per dimension): [1.04245  1.024693 1.02457  1.061017 1.098654 1.051013]
Note: some inputs slightly exceed [0, 1];
      the search for the NEXT query will be restricted to [0, 1]^6.

=== Current best (from observed data) ===
x_best = [0.019976 0.432955 0.301662 0.169496 0.348651 0.743371]
f(x_best) = 1.68082842

=== Training PyTorch ensemble surrogate ===
Training complete.

=== Proposed NEXT query point (within [0, 1]^6) ===
x_next = [0.01147806 0.62027013 0.52560605 0.05353538 0.52488092 0.66612718]
Predicted mean at x_next: 1.311853
Predictive std at x_next: 0.573189
Probability of Improvement (PI): 0.2599
Expected Improvement (EI): 0.087418

=== Agent-style Reasoning for Next Query ===
1. The current best observed output (local maxima so far) is f_best = 1.680828.
   This comes from input